# GeneTropica Phase 14 Part 2 — MD Trajectory Analysis

**Three-drug mechanism comparison on DENV NS5 RdRp (PDB 5CCV, Chain A)**

| Drug | Consensus Rank | Mechanism | Simulation |
|------|---------------|-----------|------------|
| Celecoxib | #1 (0.6846) | COX-2 inhibitor | 50 ns complete |
| Methotrexate | #3 (0.4930) | DHFR / host-directed | 50 ns complete |
| Dasabuvir | #17 (0.3758) | Non-nucleoside RdRp | 50 ns complete |

**Analyses:** RMSD, RMSF, Radius of Gyration, Hydrogen Bonds, Contact Analysis, MM-PBSA

---
Russell Young — British School Jakarta

In [ ]:
# ============================================================
# Cell 1: Install dependencies
# ============================================================
import subprocess, sys

print('Installing analysis packages...')
!pip install -q MDAnalysis matplotlib numpy pandas seaborn 2>&1 | tail -3

# Verify
for pkg in ['MDAnalysis', 'matplotlib', 'numpy', 'pandas', 'seaborn']:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', 'OK')
        print(f'  {pkg}: {ver}')
    except ImportError:
        print(f'  {pkg}: *** NOT FOUND ***')

print('\n=== Dependencies installed ===')

In [ ]:
# ============================================================
# Cell 2: Mount Google Drive & extract results
# ============================================================
import os, shutil, tarfile

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/GeneTropica_MD'
WORKDIR = '/content/md_analysis'
os.makedirs(WORKDIR, exist_ok=True)

DRUGS = ['celecoxib', 'methotrexate', 'dasabuvir']

# Extract each drug's tar.gz from Drive
for drug in DRUGS:
    tar_path = os.path.join(DRIVE_DIR, f'md_results_{drug}.tar.gz')
    drug_dir = os.path.join(WORKDIR, drug)

    if os.path.exists(os.path.join(drug_dir, 'md.xtc')):
        print(f'{drug}: already extracted')
    elif not os.path.exists(tar_path):
        # Try individual files from chunked saves
        chunk_dir = os.path.join(DRIVE_DIR, drug)
        if os.path.exists(chunk_dir):
            os.makedirs(drug_dir, exist_ok=True)
            for fn in os.listdir(chunk_dir):
                src = os.path.join(chunk_dir, fn)
                if os.path.isfile(src):
                    shutil.copy(src, drug_dir)
            print(f'{drug}: copied from Drive chunks')
        else:
            print(f'{drug}: *** NOT FOUND on Drive! ***')
            continue
    else:
        print(f'{drug}: extracting tar.gz...')
        os.makedirs(drug_dir, exist_ok=True)
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(drug_dir)

        # Flatten: if files are inside a subdirectory (e.g. md_results_celecoxib/),
        # move them up to drug_dir
        subdirs = [d for d in os.listdir(drug_dir)
                   if os.path.isdir(os.path.join(drug_dir, d))]
        for subdir in subdirs:
            subdir_path = os.path.join(drug_dir, subdir)
            for fn in os.listdir(subdir_path):
                src = os.path.join(subdir_path, fn)
                dst = os.path.join(drug_dir, fn)
                if os.path.isfile(src) and not os.path.exists(dst):
                    shutil.move(src, dst)
            if not os.listdir(subdir_path):
                os.rmdir(subdir_path)

        print(f'  Extracted to {drug_dir}')

    # After extraction, check for missing required files and look on Drive
    os.makedirs(drug_dir, exist_ok=True)
    for req_file in ['md.xtc', 'md.tpr']:
        if not os.path.exists(os.path.join(drug_dir, req_file)):
            # Search Drive for standalone copies
            search_paths = [
                os.path.join(DRIVE_DIR, drug, req_file),
                os.path.join(DRIVE_DIR, f'md_results_{drug}', req_file),
                os.path.join(DRIVE_DIR, req_file),
            ]
            for sp in search_paths:
                if os.path.exists(sp):
                    shutil.copy(sp, os.path.join(drug_dir, req_file))
                    sz = os.path.getsize(os.path.join(drug_dir, req_file)) / 1e6
                    print(f'  {drug}: found {req_file} on Drive ({sz:.1f} MB)')
                    break

# Verify key files
print('\n--- Verification ---')
REQUIRED = ['md.xtc', 'md.tpr']
OPTIONAL = ['topol.top', 'index.ndx', 'system.gro', 'md.edr', 'md.gro']
all_ok = True
for drug in DRUGS:
    drug_dir = os.path.join(WORKDIR, drug)
    print(f'\n  {drug.upper()}:')
    for fn in REQUIRED + OPTIONAL:
        path = os.path.join(drug_dir, fn)
        if os.path.exists(path):
            sz = os.path.getsize(path) / 1e6
            print(f'    {fn}: {sz:.1f} MB')
        elif fn in REQUIRED:
            print(f'    {fn}: *** MISSING (REQUIRED) ***')
            all_ok = False

if all_ok:
    print('\n=== All required files present ===')
else:
    print('\n*** Some required files are missing! ***')

In [ ]:
# ============================================================
# Cell 3: Helper functions
# ============================================================
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align, contacts
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import (
    HydrogenBondAnalysis as HBA
)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['savefig.dpi'] = 150
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

WORKDIR = '/content/md_analysis'
DRUGS = ['celecoxib', 'methotrexate', 'dasabuvir']
DRUG_COLORS = {'celecoxib': '#E74C3C', 'methotrexate': '#3498DB',
               'dasabuvir': '#2ECC71'}
DRUG_LABELS = {'celecoxib': 'Celecoxib (#1)',
               'methotrexate': 'Methotrexate (#3)',
               'dasabuvir': 'Dasabuvir (#17)'}

# Output directories
RESULTS_DIR = os.path.join(WORKDIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

DRIVE_RESULTS = '/content/drive/MyDrive/GeneTropica_MD/analysis'
os.makedirs(DRIVE_RESULTS, exist_ok=True)


def load_universe(drug):
    """Load MDAnalysis Universe for a drug."""
    drug_dir = os.path.join(WORKDIR, drug)
    tpr = os.path.join(drug_dir, 'md.tpr')
    xtc = os.path.join(drug_dir, 'md.xtc')
    u = mda.Universe(tpr, xtc)
    print(f'  {drug}: {u.trajectory.n_frames} frames, '
          f'{len(u.atoms)} atoms, '
          f'{round(u.trajectory[-1].time / 1000, 1)} ns')
    return u


def save_figure(fig, name):
    """Save figure to local results and Drive."""
    local = os.path.join(RESULTS_DIR, name)
    fig.savefig(local, bbox_inches='tight')
    shutil.copy(local, DRIVE_RESULTS)
    print(f'  Saved: {name}')


def save_csv(df, name):
    """Save DataFrame to local results and Drive."""
    local = os.path.join(RESULTS_DIR, name)
    df.to_csv(local, index=False)
    shutil.copy(local, DRIVE_RESULTS)
    print(f'  Saved: {name}')


# Load all universes
print('Loading trajectories...')
universes = {}
for drug in DRUGS:
    universes[drug] = load_universe(drug)

print('\n=== Helper functions loaded ===')

In [ ]:
# ============================================================
# Cell 4: RMSD Analysis
# ============================================================
# Protein backbone RMSD: measures overall protein stability
# Ligand RMSD: measures if drug stays in binding site

print('Computing RMSD for all 3 drugs...\n')

rmsd_data = {}

for drug in DRUGS:
    u = universes[drug]
    print(f'  {drug}...')

    # Align trajectory to first frame on protein backbone
    protein_bb = u.select_atoms('protein and backbone')
    align.AlignTraj(u, u, select='protein and backbone',
                    in_memory=False).run()

    # Protein backbone RMSD
    R_prot = rms.RMSD(u, u, select='protein and backbone',
                      ref_frame=0).run()

    # Ligand RMSD (try common residue names)
    lig_sel = None
    for sel_str in ['resname UNL', 'resname LIG', 'resname MOL',
                    'not protein and not resname SOL NA CL']:
        try:
            atoms = u.select_atoms(sel_str)
            if len(atoms) > 0 and len(atoms) < 200:
                lig_sel = sel_str
                print(f'    Ligand selection: "{sel_str}" ({len(atoms)} atoms)')
                break
        except:
            continue

    R_lig = None
    if lig_sel:
        try:
            R_lig = rms.RMSD(u, u, select=lig_sel, ref_frame=0).run()
        except Exception as e:
            print(f'    Ligand RMSD failed: {e}')

    time_ns = R_prot.results.rmsd[:, 1] / 1000  # ps -> ns
    prot_rmsd = R_prot.results.rmsd[:, 2]  # Angstrom
    lig_rmsd = R_lig.results.rmsd[:, 2] if R_lig else np.zeros_like(prot_rmsd)

    rmsd_data[drug] = {
        'time_ns': time_ns,
        'protein': prot_rmsd,
        'ligand': lig_rmsd
    }

    # Save per-drug CSV
    df = pd.DataFrame({
        'time_ns': time_ns,
        'protein_rmsd_A': prot_rmsd,
        'ligand_rmsd_A': lig_rmsd
    })
    save_csv(df, f'rmsd_{drug}.csv')

    print(f'    Protein RMSD: {prot_rmsd[-100:].mean():.2f} +/- '
          f'{prot_rmsd[-100:].std():.2f} A (last 5 ns)')
    if R_lig:
        print(f'    Ligand RMSD:  {lig_rmsd[-100:].mean():.2f} +/- '
              f'{lig_rmsd[-100:].std():.2f} A (last 5 ns)')

# --- Comparison plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Protein RMSD
for drug in DRUGS:
    axes[0].plot(rmsd_data[drug]['time_ns'], rmsd_data[drug]['protein'],
                 color=DRUG_COLORS[drug], label=DRUG_LABELS[drug],
                 linewidth=0.8, alpha=0.8)
axes[0].set_xlabel('Time (ns)')
axes[0].set_ylabel('RMSD (\u00c5)')
axes[0].set_title('Protein Backbone RMSD')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Ligand RMSD
for drug in DRUGS:
    axes[1].plot(rmsd_data[drug]['time_ns'], rmsd_data[drug]['ligand'],
                 color=DRUG_COLORS[drug], label=DRUG_LABELS[drug],
                 linewidth=0.8, alpha=0.8)
axes[1].set_xlabel('Time (ns)')
axes[1].set_ylabel('RMSD (\u00c5)')
axes[1].set_title('Ligand RMSD')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

fig.suptitle('RMSD Analysis — 50 ns MD Simulation', fontsize=13,
             fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'rmsd_comparison.png')
plt.show()

print('\n=== RMSD analysis complete ===')

In [ ]:
# ============================================================
# Cell 5: RMSF Analysis
# ============================================================
# Per-residue flexibility — identifies mobile regions

print('Computing RMSF for all 3 drugs...\n')

rmsf_data = {}

for drug in DRUGS:
    u = universes[drug]
    print(f'  {drug}...')

    # Align to average structure
    protein_ca = u.select_atoms('protein and name CA')
    align.AlignTraj(u, u, select='protein and name CA',
                    in_memory=False).run()

    # Compute RMSF per C-alpha
    R = rms.RMSF(protein_ca).run()

    resids = protein_ca.resids
    rmsf_vals = R.results.rmsf

    rmsf_data[drug] = {'resids': resids, 'rmsf': rmsf_vals}

    df = pd.DataFrame({'resid': resids, 'rmsf_A': rmsf_vals})
    save_csv(df, f'rmsf_{drug}.csv')

    # Top 10 most flexible residues
    top_idx = np.argsort(rmsf_vals)[-10:][::-1]
    print(f'    Top flexible residues: '
          f'{["%d (%.1f A)" % (resids[i], rmsf_vals[i]) for i in top_idx[:5]]}')

# --- Comparison plot ---
fig, ax = plt.subplots(figsize=(14, 5))

for drug in DRUGS:
    ax.plot(rmsf_data[drug]['resids'], rmsf_data[drug]['rmsf'],
            color=DRUG_COLORS[drug], label=DRUG_LABELS[drug],
            linewidth=0.8, alpha=0.7)

ax.set_xlabel('Residue Number')
ax.set_ylabel('RMSF (\u00c5)')
ax.set_title('Per-Residue RMSF — C\u03b1 Atoms', fontsize=13,
             fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
save_figure(fig, 'rmsf_comparison.png')
plt.show()

print('\n=== RMSF analysis complete ===')

In [ ]:
# ============================================================
# Cell 6: Radius of Gyration
# ============================================================
# Protein compactness — checks for unfolding

print('Computing Radius of Gyration...\n')

rg_data = {}

for drug in DRUGS:
    u = universes[drug]
    print(f'  {drug}...')

    protein = u.select_atoms('protein')
    rg_list = []
    time_list = []

    for ts in u.trajectory:
        rg_list.append(protein.radius_of_gyration())
        time_list.append(ts.time / 1000)  # ps -> ns

    rg_data[drug] = {'time_ns': np.array(time_list),
                     'rg': np.array(rg_list)}

    rg_arr = np.array(rg_list)
    print(f'    Rg: {rg_arr.mean():.2f} +/- {rg_arr.std():.2f} A')

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 5))

for drug in DRUGS:
    ax.plot(rg_data[drug]['time_ns'], rg_data[drug]['rg'],
            color=DRUG_COLORS[drug], label=DRUG_LABELS[drug],
            linewidth=0.8, alpha=0.8)

ax.set_xlabel('Time (ns)')
ax.set_ylabel('Radius of Gyration (\u00c5)')
ax.set_title('Protein Radius of Gyration — Compactness Check',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
save_figure(fig, 'rg_comparison.png')
plt.show()

print('\n=== Radius of Gyration analysis complete ===')

In [ ]:
# ============================================================
# Cell 7: Hydrogen Bond Analysis
# ============================================================
# Drug-protein hydrogen bonds — key binding interactions

print('Computing hydrogen bonds for all 3 drugs...\n')

hbond_data = {}

for drug in DRUGS:
    u = universes[drug]
    print(f'  {drug}...')

    # Find ligand selection
    lig_sel = None
    for sel_str in ['resname UNL', 'resname LIG', 'resname MOL',
                    'not protein and not resname SOL NA CL']:
        try:
            atoms = u.select_atoms(sel_str)
            if len(atoms) > 0 and len(atoms) < 200:
                lig_sel = sel_str
                break
        except:
            continue

    if not lig_sel:
        print(f'    Could not find ligand atoms!')
        continue

    # H-bonds: protein as donor/acceptor, ligand as acceptor/donor
    hbonds = HBA(
        universe=u,
        donors_sel=f'protein',
        acceptors_sel=lig_sel,
        d_a_cutoff=3.5,
        d_h_a_angle_cutoff=120,
    )
    hbonds.run()

    # Also check ligand as donor to protein
    hbonds2 = HBA(
        universe=u,
        donors_sel=lig_sel,
        acceptors_sel='protein',
        d_a_cutoff=3.5,
        d_h_a_angle_cutoff=120,
    )
    hbonds2.run()

    # Count per frame
    n_frames = u.trajectory.n_frames
    counts = np.zeros(n_frames)

    if len(hbonds.results.hbonds) > 0:
        for hb in hbonds.results.hbonds:
            frame_idx = int(hb[0])
            if frame_idx < n_frames:
                counts[frame_idx] += 1

    if len(hbonds2.results.hbonds) > 0:
        for hb in hbonds2.results.hbonds:
            frame_idx = int(hb[0])
            if frame_idx < n_frames:
                counts[frame_idx] += 1

    time_ns = np.arange(n_frames) * 0.05  # 50 ps per frame

    hbond_data[drug] = {
        'time_ns': time_ns,
        'counts': counts,
        'hbonds_1': hbonds.results.hbonds,
        'hbonds_2': hbonds2.results.hbonds
    }

    df = pd.DataFrame({'time_ns': time_ns, 'n_hbonds': counts})
    save_csv(df, f'hbonds_{drug}.csv')

    print(f'    Average H-bonds: {counts.mean():.1f} +/- {counts.std():.1f}')
    print(f'    Max H-bonds in single frame: {int(counts.max())}')

    # Identify persistent H-bonds (occupancy > 30%)
    all_hb = np.vstack([hbonds.results.hbonds, hbonds2.results.hbonds]) \
             if len(hbonds2.results.hbonds) > 0 and len(hbonds.results.hbonds) > 0 \
             else (hbonds.results.hbonds if len(hbonds.results.hbonds) > 0 \
                   else hbonds2.results.hbonds)

    if len(all_hb) > 0:
        # Group by donor-acceptor pair
        pair_counts = {}
        for hb in all_hb:
            donor_idx = int(hb[1])
            acceptor_idx = int(hb[3])
            try:
                donor_atom = u.atoms[donor_idx]
                acceptor_atom = u.atoms[acceptor_idx]
                pair_key = (f'{donor_atom.resname}{donor_atom.resid}-{donor_atom.name}',
                            f'{acceptor_atom.resname}{acceptor_atom.resid}-{acceptor_atom.name}')
                pair_counts[pair_key] = pair_counts.get(pair_key, 0) + 1
            except:
                continue

        # Show top persistent H-bonds
        sorted_pairs = sorted(pair_counts.items(), key=lambda x: -x[1])
        print(f'    Persistent H-bonds (>30% occupancy):')
        for (donor, acceptor), count in sorted_pairs[:10]:
            occ = count / n_frames * 100
            if occ > 30:
                print(f'      {donor} -> {acceptor}: {occ:.1f}%')

# --- Comparison plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time series
for drug in DRUGS:
    if drug in hbond_data:
        # Running average for clarity
        window = 20
        counts_smooth = np.convolve(hbond_data[drug]['counts'],
                                    np.ones(window)/window, mode='valid')
        time_smooth = hbond_data[drug]['time_ns'][:len(counts_smooth)]
        axes[0].plot(time_smooth, counts_smooth,
                     color=DRUG_COLORS[drug], label=DRUG_LABELS[drug],
                     linewidth=0.8, alpha=0.8)

axes[0].set_xlabel('Time (ns)')
axes[0].set_ylabel('Number of H-bonds')
axes[0].set_title('Drug-Protein H-bonds (running avg)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Bar chart of averages
means = [hbond_data[d]['counts'].mean() for d in DRUGS if d in hbond_data]
stds = [hbond_data[d]['counts'].std() for d in DRUGS if d in hbond_data]
labels = [DRUG_LABELS[d] for d in DRUGS if d in hbond_data]
colors = [DRUG_COLORS[d] for d in DRUGS if d in hbond_data]
axes[1].bar(range(len(means)), means, yerr=stds, color=colors,
            capsize=5, alpha=0.8)
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels([d.split(' ')[0] for d in labels], fontsize=9)
axes[1].set_ylabel('Average H-bonds')
axes[1].set_title('Mean H-bonds per Drug')
axes[1].grid(True, alpha=0.3, axis='y')

fig.suptitle('Hydrogen Bond Analysis', fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'hbonds_comparison.png')
plt.show()

print('\n=== Hydrogen bond analysis complete ===')

In [ ]:
# ============================================================
# Cell 8: Contact Analysis
# ============================================================
# Which protein residues are in contact with each drug?

print('Computing drug-protein contacts...\n')

contact_data = {}
CUTOFF = 4.5  # Angstrom

for drug in DRUGS:
    u = universes[drug]
    print(f'  {drug}...')

    # Find ligand
    lig_sel = None
    for sel_str in ['resname UNL', 'resname LIG', 'resname MOL',
                    'not protein and not resname SOL NA CL']:
        try:
            atoms = u.select_atoms(sel_str)
            if len(atoms) > 0 and len(atoms) < 200:
                lig_sel = sel_str
                break
        except:
            continue

    if not lig_sel:
        print(f'    Could not find ligand!')
        continue

    ligand = u.select_atoms(lig_sel)
    protein = u.select_atoms('protein')

    # Count contacts per residue across all frames
    residue_contacts = {}
    n_frames = u.trajectory.n_frames

    for ts in u.trajectory:
        # Find protein atoms within CUTOFF of any ligand atom
        nearby = u.select_atoms(
            f'protein and around {CUTOFF} ({lig_sel})',
            updating=True
        )
        for res in set(nearby.resids):
            residue_contacts[res] = residue_contacts.get(res, 0) + 1

    # Convert to occupancy percentage
    contact_freq = {res: count / n_frames * 100
                    for res, count in residue_contacts.items()}

    # Sort by frequency
    sorted_contacts = sorted(contact_freq.items(), key=lambda x: -x[1])

    contact_data[drug] = sorted_contacts

    df = pd.DataFrame(sorted_contacts, columns=['resid', 'occupancy_pct'])
    save_csv(df, f'contacts_{drug}.csv')

    print(f'    Residues in contact (>50% occupancy):')
    for resid, occ in sorted_contacts[:15]:
        if occ > 50:
            # Get residue name
            try:
                resname = protein.select_atoms(f'resid {resid}').residues[0].resname
                print(f'      {resname} {resid}: {occ:.1f}%')
            except:
                print(f'      Res {resid}: {occ:.1f}%')

# --- Comparison: Top 20 contact residues per drug ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, drug in enumerate(DRUGS):
    if drug not in contact_data:
        continue
    top20 = contact_data[drug][:20]
    if not top20:
        continue
    resids = [str(r) for r, _ in top20]
    occs = [o for _, o in top20]

    axes[idx].barh(range(len(resids)), occs, color=DRUG_COLORS[drug],
                   alpha=0.8)
    axes[idx].set_yticks(range(len(resids)))
    axes[idx].set_yticklabels(resids, fontsize=8)
    axes[idx].set_xlabel('Contact Occupancy (%)')
    axes[idx].set_title(DRUG_LABELS[drug], fontsize=10)
    axes[idx].invert_yaxis()
    axes[idx].grid(True, alpha=0.3, axis='x')

fig.suptitle(f'Top 20 Contact Residues (cutoff {CUTOFF} \u00c5)',
             fontsize=13, fontweight='bold')
fig.tight_layout()
save_figure(fig, 'contacts_comparison.png')
plt.show()

print('\n=== Contact analysis complete ===')

In [ ]:
# ============================================================
# Cell 9: MM-PBSA Binding Free Energy (Approximation)
# ============================================================
# Estimates binding energy using MDAnalysis interaction energies.
# Full MM-PBSA requires gmx_MMPBSA — this is a lightweight
# approximation using short-range nonbonded interaction energy.

print('Computing interaction energies (MM-PBSA approximation)...\n')
print('Note: This uses short-range interaction energy as a proxy.')
print('For publication-grade MM-PBSA, use gmx_MMPBSA separately.\n')

# Try to install gmx_MMPBSA for full calculation
try:
    import subprocess
    r = subprocess.run('pip install gmx_MMPBSA 2>&1 | tail -3',
                       shell=True, capture_output=True, text=True)
    from GMXMMPBSA import API as gmxapi
    HAS_MMPBSA = True
    print('gmx_MMPBSA: installed')
except:
    HAS_MMPBSA = False
    print('gmx_MMPBSA: not available, using interaction energy proxy')

ie_data = {}

for drug in DRUGS:
    u = universes[drug]
    print(f'\n  {drug}...')

    # Find ligand
    lig_sel = None
    for sel_str in ['resname UNL', 'resname LIG', 'resname MOL',
                    'not protein and not resname SOL NA CL']:
        try:
            atoms = u.select_atoms(sel_str)
            if len(atoms) > 0 and len(atoms) < 200:
                lig_sel = sel_str
                break
        except:
            continue

    if not lig_sel:
        print(f'    Could not find ligand!')
        continue

    # Use minimum distance between protein and ligand as stability proxy
    # along with number of contacts as binding strength indicator
    ligand = u.select_atoms(lig_sel)
    protein = u.select_atoms('protein')

    min_dists = []
    n_contacts_list = []
    time_list = []

    # Sample every 5th frame for speed
    for i, ts in enumerate(u.trajectory):
        if i % 5 != 0:
            continue
        # Minimum distance between ligand and protein
        from MDAnalysis.lib.distances import distance_array
        dists = distance_array(ligand.positions, protein.positions)
        min_dists.append(dists.min())
        # Contact count within 4.5 A
        n_contacts_list.append((dists < 4.5).sum())
        time_list.append(ts.time / 1000)

    min_dists = np.array(min_dists)
    n_contacts_arr = np.array(n_contacts_list)
    time_arr = np.array(time_list)

    ie_data[drug] = {
        'time_ns': time_arr,
        'min_dist': min_dists,
        'n_contacts': n_contacts_arr
    }

    print(f'    Min distance to protein: {min_dists.mean():.2f} +/- '
          f'{min_dists.std():.2f} A')
    print(f'    Avg atom-atom contacts (<4.5A): '
          f'{n_contacts_arr.mean():.0f} +/- {n_contacts_arr.std():.0f}')

    df = pd.DataFrame({
        'time_ns': time_arr,
        'min_dist_A': min_dists,
        'n_contacts': n_contacts_arr
    })
    save_csv(df, f'binding_proxy_{drug}.csv')

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for drug in DRUGS:
    if drug not in ie_data:
        continue
    axes[0].plot(ie_data[drug]['time_ns'], ie_data[drug]['min_dist'],
                 color=DRUG_COLORS[drug], label=DRUG_LABELS[drug],
                 linewidth=0.8, alpha=0.8)
    axes[1].plot(ie_data[drug]['time_ns'], ie_data[drug]['n_contacts'],
                 color=DRUG_COLORS[drug], label=DRUG_LABELS[drug],
                 linewidth=0.8, alpha=0.8)

axes[0].set_xlabel('Time (ns)')
axes[0].set_ylabel('Minimum Distance (\u00c5)')
axes[0].set_title('Ligand-Protein Minimum Distance')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Time (ns)')
axes[1].set_ylabel('Number of Atom-Atom Contacts')
axes[1].set_title('Ligand-Protein Contacts (<4.5 \u00c5)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

fig.suptitle('Binding Stability Proxy', fontsize=13,
             fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'binding_proxy_comparison.png')
plt.show()

print('\n=== Binding energy proxy analysis complete ===')

In [ ]:
# ============================================================
# Cell 10: Three-Drug Comparison Summary
# ============================================================

print('Generating comparison summary...\n')

summary_rows = []

for drug in DRUGS:
    row = {'Drug': drug.capitalize()}

    # RMSD (last 10 ns average)
    if drug in rmsd_data:
        prot = rmsd_data[drug]['protein'][-200:]  # last 10 ns
        lig = rmsd_data[drug]['ligand'][-200:]
        row['Prot_RMSD_avg'] = f'{prot.mean():.2f}'
        row['Prot_RMSD_std'] = f'{prot.std():.2f}'
        row['Lig_RMSD_avg'] = f'{lig.mean():.2f}'
        row['Lig_RMSD_std'] = f'{lig.std():.2f}'

    # Rg
    if drug in rg_data:
        rg = rg_data[drug]['rg']
        row['Rg_avg'] = f'{rg.mean():.2f}'

    # H-bonds
    if drug in hbond_data:
        hb = hbond_data[drug]['counts']
        row['HBonds_avg'] = f'{hb.mean():.1f}'
        row['HBonds_std'] = f'{hb.std():.1f}'

    # Contacts
    if drug in contact_data:
        n_res = sum(1 for _, occ in contact_data[drug] if occ > 50)
        row['ContactRes_gt50pct'] = str(n_res)

    # Binding proxy
    if drug in ie_data:
        row['MinDist_avg'] = f"{ie_data[drug]['min_dist'].mean():.2f}"
        row['Contacts_avg'] = f"{ie_data[drug]['n_contacts'].mean():.0f}"

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
save_csv(summary_df, 'comparison_summary.csv')

print('\n' + '='*70)
print('  THREE-DRUG COMPARISON SUMMARY (50 ns MD)')
print('='*70)
print(summary_df.to_string(index=False))
print('='*70)

# --- Multi-panel comparison figure ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Protein RMSD
for drug in DRUGS:
    if drug in rmsd_data:
        axes[0,0].plot(rmsd_data[drug]['time_ns'],
                       rmsd_data[drug]['protein'],
                       color=DRUG_COLORS[drug], label=DRUG_LABELS[drug],
                       linewidth=0.6, alpha=0.7)
axes[0,0].set_title('Protein Backbone RMSD')
axes[0,0].set_xlabel('Time (ns)')
axes[0,0].set_ylabel('RMSD (\u00c5)')
axes[0,0].legend(fontsize=7)
axes[0,0].grid(True, alpha=0.3)

# 2. Ligand RMSD
for drug in DRUGS:
    if drug in rmsd_data:
        axes[0,1].plot(rmsd_data[drug]['time_ns'],
                       rmsd_data[drug]['ligand'],
                       color=DRUG_COLORS[drug], label=DRUG_LABELS[drug],
                       linewidth=0.6, alpha=0.7)
axes[0,1].set_title('Ligand RMSD')
axes[0,1].set_xlabel('Time (ns)')
axes[0,1].set_ylabel('RMSD (\u00c5)')
axes[0,1].legend(fontsize=7)
axes[0,1].grid(True, alpha=0.3)

# 3. Radius of Gyration
for drug in DRUGS:
    if drug in rg_data:
        axes[0,2].plot(rg_data[drug]['time_ns'], rg_data[drug]['rg'],
                       color=DRUG_COLORS[drug], label=DRUG_LABELS[drug],
                       linewidth=0.6, alpha=0.7)
axes[0,2].set_title('Radius of Gyration')
axes[0,2].set_xlabel('Time (ns)')
axes[0,2].set_ylabel('Rg (\u00c5)')
axes[0,2].legend(fontsize=7)
axes[0,2].grid(True, alpha=0.3)

# 4. H-bonds (bar)
hb_means = [hbond_data[d]['counts'].mean() for d in DRUGS if d in hbond_data]
hb_stds = [hbond_data[d]['counts'].std() for d in DRUGS if d in hbond_data]
hb_colors = [DRUG_COLORS[d] for d in DRUGS if d in hbond_data]
hb_labels = [d.capitalize() for d in DRUGS if d in hbond_data]
axes[1,0].bar(range(len(hb_means)), hb_means, yerr=hb_stds,
              color=hb_colors, capsize=5, alpha=0.8)
axes[1,0].set_xticks(range(len(hb_labels)))
axes[1,0].set_xticklabels(hb_labels)
axes[1,0].set_title('Avg Drug-Protein H-bonds')
axes[1,0].set_ylabel('H-bonds')
axes[1,0].grid(True, alpha=0.3, axis='y')

# 5. Contact residues (bar)
ct_vals = [sum(1 for _, o in contact_data[d] if o > 50)
           for d in DRUGS if d in contact_data]
ct_colors = [DRUG_COLORS[d] for d in DRUGS if d in contact_data]
ct_labels = [d.capitalize() for d in DRUGS if d in contact_data]
axes[1,1].bar(range(len(ct_vals)), ct_vals, color=ct_colors, alpha=0.8)
axes[1,1].set_xticks(range(len(ct_labels)))
axes[1,1].set_xticklabels(ct_labels)
axes[1,1].set_title('Contact Residues (>50% occ.)')
axes[1,1].set_ylabel('Number of Residues')
axes[1,1].grid(True, alpha=0.3, axis='y')

# 6. Min distance (bar)
md_means = [ie_data[d]['min_dist'].mean() for d in DRUGS if d in ie_data]
md_stds = [ie_data[d]['min_dist'].std() for d in DRUGS if d in ie_data]
md_colors = [DRUG_COLORS[d] for d in DRUGS if d in ie_data]
md_labels = [d.capitalize() for d in DRUGS if d in ie_data]
axes[1,2].bar(range(len(md_means)), md_means, yerr=md_stds,
              color=md_colors, capsize=5, alpha=0.8)
axes[1,2].set_xticks(range(len(md_labels)))
axes[1,2].set_xticklabels(md_labels)
axes[1,2].set_title('Avg Min Distance to Protein')
axes[1,2].set_ylabel('Distance (\u00c5)')
axes[1,2].grid(True, alpha=0.3, axis='y')

fig.suptitle('Three-Drug Mechanism Comparison — 50 ns MD on DENV NS5 RdRp',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'comparison_figure.png')
plt.show()

print('\n=== Comparison summary complete ===')

In [ ]:
# ============================================================
# Cell 11: Save all results to Drive
# ============================================================
import tarfile
from google.colab import files

print('Packaging all results...\n')

# Create tar.gz of all results
tar_path = os.path.join(WORKDIR, 'md_analysis_results.tar.gz')
with tarfile.open(tar_path, 'w:gz') as tar:
    for fn in sorted(os.listdir(RESULTS_DIR)):
        full = os.path.join(RESULTS_DIR, fn)
        tar.add(full, arcname=fn)
        print(f'  Added: {fn}')

size_mb = os.path.getsize(tar_path) / 1e6
print(f'\nPackage: md_analysis_results.tar.gz ({size_mb:.1f} MB)')

# Save to Drive
drive_ok = False
if os.path.exists('/content/drive/MyDrive'):
    shutil.copy(tar_path, DRIVE_RESULTS)
    print(f'Saved to Drive: {DRIVE_RESULTS}/md_analysis_results.tar.gz')
    drive_ok = True
else:
    print('WARNING: Drive not mounted!')

# Browser download
try:
    files.download(tar_path)
    print('Browser download started')
except Exception as e:
    if drive_ok:
        print(f'(Browser download skipped — file is safe on Drive)')
    else:
        print(f'Download failed: {e}')

# Final summary
print('\n' + '='*60)
print('  PHASE 14 PART 2 — MD ANALYSIS COMPLETE')
print('='*60)
print(f'\nResults on Drive: {DRIVE_RESULTS}/')
print('\nFiles generated:')
for fn in sorted(os.listdir(RESULTS_DIR)):
    sz = os.path.getsize(os.path.join(RESULTS_DIR, fn)) / 1e3
    print(f'  {fn} ({sz:.1f} KB)')
print('\nExtract md_analysis_results.tar.gz and place contents in:')
print('  data/md_simulation/comparison/')
print('\nReady for paper/report writing!')